<a href="https://colab.research.google.com/github/jameshphan-png/Group-Exercise-Agentic-AI-in-Marketing-/blob/dev/Agentic_AI_in_Marketing_Group_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IMPORTANT: REFER TO THE README.MD FILE ON GITHUB FOR INSTRUCTIONS**
- Link: https://github.com/jameshphan-png/Group-Exercise-Agentic-AI-in-Marketing-/blob/dev/README.md

# Ad Optimization Agent

This is an Ad Optimization Agent where it allocates budgets across all 3 channels
- **Search (Google)**
- **Social (Facebook)**
- **Display (Sprouts)**

Notebook includes:
- **LangGraph**
- **LangChain**
- **tools**
- **AgentState**
- **graph nodes**
- **baseline vs agent evaluation**


In [ ]:
# Install required packages.
# In Colab or a fresh notebook, run this once before the rest of the notebook.
%pip install -q langgraph langchain langchain-openai openai pandas langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00


-----

### 1. Imports and State Definition

**Description:**  
This block brings in the libraries needed for the agent. It also defines the shared `AgentState`, which is the memory object that moves through the LangGraph workflow.

**Why the AgentState matters:**  
This uses a stateful graph. That means each node should be able to read what happened before and write back new information. In this assignment, the state stores:
- the raw CSV data
- daily channel metrics
- budget allocations
- decisions and reason strings
- evaluation results


In [ ]:
from __future__ import annotations

# Standard Python libraries
import json
import math
import os
os.environ["GOOGLE_API_KEY"] = "test3"
from io import StringIO
from typing import Dict, List, Literal, TypedDict

# Data work
import pandas as pd

# Structured outputs for the LLM
from pydantic import BaseModel, Field

# LangChain pieces
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# LangGraph workflow pieces
from langgraph.graph import END, StateGraph


# ---------------------------------------------------------
# AgentState:
# This is the central memory object that flows through the graph.
# Every node reads from this state and returns updates to it.
# ---------------------------------------------------------
class AgentState(TypedDict, total=False):
    system_prompt: str
    raw_data: pd.DataFrame
    daily_data: Dict[str, pd.DataFrame]
    dates: List[str]
    current_day_index: int
    current_allocations: Dict[str, float]
    next_allocations: Dict[str, float]
    latest_day_metrics: Dict[str, Dict[str, float]]
    latest_llm_decision: Dict[str, object]
    latest_tool_actions: List[Dict[str, object]]
    decision_history: List[Dict[str, object]]
    agent_daily_results: List[Dict[str, object]]
    baseline_daily_results: List[Dict[str, object]]
    evaluation_summary: Dict[str, Dict[str, float]]

-----

### 2. System Prompt, Settings, and API Setup

**Description:**  
This is the instruction layer for the agent. It explains what the agent is trying to do and what rules it must follow.

**Settings Feature:**
- `DEBUG_MODE`: test the full graph without making repeated live API calls
- `DAILY_SHIFT`: controls how much budget moves toward the top performer
- `MAX_DAILY_CHANGE`: prevents aggressive budget swings
- `MIN_CHANNEL_FLOOR`: keeps every channel active for learning
- `llm`: connects LangChain to the Gemini API endpoint


In [ ]:
# ---------------------------------------------------------
# Debug mode:
# True  -> use local heuristic reasoning inside the reasoning node
# False -> use live LLM calls through GeminiAPI
#
# Precautionary measures for API limits when testing
# ---------------------------------------------------------
DEBUG_MODE = True

# Optional: reduce the number of days while testing
# Example: MAX_DEBUG_DAYS = 5
MAX_DEBUG_DAYS = None


# ---------------------------------------------------------
# System prompt:
# This tells the model how to behave when we use the LLM path.
# ---------------------------------------------------------
SYSTEM_PROMPT = """
You are a lightweight Ad Optimization Agent for marketing budget allocation.

Your job:
- Read daily performance data from Search, Social, and Display channels.
- Learn from recent data using simple reasoning.
- Reallocate the next day's budget to improve conversions while also tracking CTR.

Rules you must follow:
1. Start with an equal split across Search, Social, and Display.
2. Use a simple explore/exploit approach:
   - reward the strongest channel by shifting 10% of budget toward it
   - keep budget on the other channels so learning continues
3. Apply guardrails:
   - cap daily budget change at plus or minus 20%
   - keep at least a 20% minimum budget floor on every channel
   - never fully shut off a channel
4. Use plain-English reason strings for every decision.
5. If one channel is weak, suggest helpful actions such as:
   - adjust_bid
   - pause_ad_group
   - request_new_creatives
6. Return decisions that are easy to audit and explain.
""".strip()


# ---------------------------------------------------------
# Global constants used by the budget logic
# ---------------------------------------------------------
CHANNELS = ["Search", "Social", "Display"]
DAILY_SHIFT = 0.10
MAX_DAILY_CHANGE = 0.20
MIN_CHANNEL_FLOOR = 0.20


# ---------------------------------------------------------
# LLM setup:
# Using Google Gemini via LangChain
# ---------------------------------------------------------

# Install once if needed:
# %pip install -q langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI

# Set your API key (make sure this is set before running)
# Example:
# import os
# os.environ["GOOGLE_API_KEY"] = "your_api_key_here"

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",  # fast + free-tier friendly
    temperature=0
)

print(f"DEBUG_MODE = {DEBUG_MODE}")

DEBUG_MODE = True


-----

### 3. Structured LLM Output Schema

**Description:**  
When the LLM is used, we want it to return a clean and predictable answer instead of messy free text.

**Classes:**  
`BudgetDecision` defines the exact structure we expect from the model:
- which channel should get more budget
- what metric was used
- why that choice was made
- what follow-up actions are recommended


In [ ]:
class BudgetDecision(BaseModel):
    # The single best channel for the next budget increase
    top_channel: Literal["Search", "Social", "Display"] = Field(
        description="The channel that should receive more budget tomorrow."
    )

    # The main metric the model relied on
    metric_used: Literal["CTR", "CVR", "CPA", "blended_score"] = Field(
        description="The main metric used to justify the choice."
    )

    # One short explanation in plain English
    reasoning: str = Field(
        description="One short plain-English explanation of the decision."
    )

    # Optional simulated actions the agent may want to take
    suggested_actions: List[str] = Field(
        default_factory=list,
        description="Optional tool actions such as adjust_bid, pause_ad_group, request_new_creatives."
    )

-----

### 4. Define Custom Tools

**Description:**  
The template highlights that tools are Python functions the agent can use to interact with the outside world.

In this assignment, the tools are simulated, but they still mirror real ad-ops actions:
- `read_performance_data` reads the CSV inputs
- `calculate_channel_metrics` calculates CTR, CVR, CPA, and CPC
- `adjust_bid` simulates increasing or decreasing bids
- `pause_ad_group` simulates pausing a weak group
- `request_new_creatives` simulates asking for better ad creative

**What “tool” means here:**  
A tool is a normal Python function wrapped with `@tool` so LangChain can call it in a tool-like way.


In [ ]:
@tool
def read_performance_data(file_names: List[str]) -> str:
    """
    Reads multiple CSV files and combines them into one dataset.

    Expected columns:
    - date
    - channel
    - spend
    - impressions
    - clicks
    - conversions

    Returns:
    A JSON string so the data can move easily through the graph.
    """
    frames = []
    expected = ["date", "channel", "spend", "impressions", "clicks", "conversions"]

    # Read each file, standardize the columns, and validate the schema.
    for file_name in file_names:
        df = pd.read_csv(file_name)
        df.columns = [c.strip().lower() for c in df.columns]

        # Make sure the CSV matches the assignment requirements.
        missing = [c for c in expected if c not in df.columns]
        if missing:
            raise ValueError(f"{file_name} is missing columns: {missing}")

        # Keep only the required columns in a consistent order.
        df = df[expected].copy()
        df["date"] = pd.to_datetime(df["date"])
        df["channel"] = df["channel"].astype(str).str.strip().str.title()
        frames.append(df)

    # Merge all channel files into one combined dataset.
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["date", "channel"]).reset_index(drop=True)

    return combined.to_json(orient="records", date_format="iso")


@tool
def calculate_channel_metrics(day_json: str) -> str:
    """
    Calculates daily channel performance metrics.

    Metrics:
    - CTR = clicks / impressions
    - CVR = conversions / clicks
    - CPA = spend / conversions
    - CPC = spend / clicks

    Also returns a blended score so the agent can compare channels
    using more than one metric at once.
    """
    # StringIO removes the FutureWarning from pd.read_json on literal strings.
    day_df = pd.read_json(StringIO(day_json))
    metrics = {}

    for _, row in day_df.iterrows():
        channel = row["channel"]
        spend = float(row["spend"])
        impressions = max(float(row["impressions"]), 1.0)
        clicks = max(float(row["clicks"]), 0.0)
        conversions = max(float(row["conversions"]), 0.0)

        # Core performance formulas
        ctr = clicks / impressions if impressions else 0.0
        cvr = conversions / clicks if clicks else 0.0
        cpa = spend / conversions if conversions else float("inf")
        cpc = spend / clicks if clicks else float("inf")

        # Blended score:
        # This gives more weight to conversion efficiency, while still
        # considering engagement and cost efficiency.
        blended_score = (
            0.45 * cvr
            + 0.35 * ctr
            + 0.20 * (1.0 / cpa if math.isfinite(cpa) and cpa > 0 else 0.0)
        )

        metrics[channel] = {
            "spend": round(spend, 2),
            "impressions": int(impressions),
            "clicks": int(clicks),
            "conversions": int(conversions),
            "ctr": ctr,
            "cvr": cvr,
            "cpa": None if not math.isfinite(cpa) else cpa,
            "cpc": None if not math.isfinite(cpc) else cpc,
            "blended_score": blended_score,
        }

    return json.dumps(metrics)


@tool
def adjust_bid(channel: str, direction: str, percent: float, reason: str) -> str:
    """
    Simulates a bid adjustment action.

    In a real ad platform, this could increase or decrease bids
    for a campaign or ad group.
    """
    return json.dumps(
        {
            "tool": "adjust_bid",
            "channel": channel,
            "direction": direction,
            "percent": percent,
            "reason": reason,
            "status": "simulated",
        }
    )


@tool
def pause_ad_group(channel: str, reason: str) -> str:
    """
    Simulates pausing a weak ad group.

    This is useful when a channel has poor conversion quality
    and the agent wants to reduce waste.
    """
    return json.dumps(
        {
            "tool": "pause_ad_group",
            "channel": channel,
            "reason": reason,
            "status": "simulated",
        }
    )


@tool
def request_new_creatives(channel: str, reason: str) -> str:
    """
    Simulates requesting new creative assets.

    This is useful when CTR is weak and the ads may need better copy
    or visuals.
    """
    return json.dumps(
        {
            "tool": "request_new_creatives",
            "channel": channel,
            "reason": reason,
            "status": "simulated",
        }
    )

-----

### 5. Helper Functions

**Description:**  
These helper functions keep the graph nodes cleaner and easier to understand. Rulesets are enforced here.

**What each helper means:**
- `normalize_allocations`: makes sure allocations sum to 100%
- `apply_guardrails`: enforces the daily budget rules
- `heuristic_budget_shift`: moves extra budget to the top performer
- `simulate_day_outcome`: estimates what happens after the budget shift
- `summarize_results`: creates final evaluation metrics
- formatting helpers make the output easier to read


In [ ]:
def normalize_allocations(allocations: Dict[str, float]) -> Dict[str, float]:
    """
    Rescales the allocations so the total always adds up to 1.0 (100%).
    """
    total = sum(allocations.values())
    return {k: v / total for k, v in allocations.items()}


def apply_guardrails(
    old_allocations: Dict[str, float],
    proposed_allocations: Dict[str, float],
    min_floor: float = MIN_CHANNEL_FLOOR,
    max_change: float = MAX_DAILY_CHANGE,
) -> Dict[str, float]:
    """
    Applies safety rules to the proposed budget split.

    Guardrails used:
    - no channel drops below the minimum floor
    - no channel changes by more than the max daily change
    - allocations are renormalized to 100%
    """
    adjusted = {}

    for channel in CHANNELS:
        old_value = old_allocations[channel]

        # Lower and upper bounds for each channel's next allocation.
        lower = max(min_floor, old_value - max_change)
        upper = min(1.0, old_value + max_change)

        # Clamp the proposed value inside the allowed range.
        adjusted[channel] = min(max(proposed_allocations[channel], lower), upper)

    adjusted = normalize_allocations(adjusted)

    # Safety re-check to keep every channel above the minimum floor.
    for channel in CHANNELS:
        if adjusted[channel] < min_floor:
            adjusted[channel] = min_floor

    return normalize_allocations(adjusted)


def heuristic_budget_shift(
    current_allocations: Dict[str, float],
    top_channel: str,
    shift_pct: float = DAILY_SHIFT,
) -> Dict[str, float]:
    """
    Implements the simple explore/exploit baseline rule.

    Logic:
    - move `shift_pct` more budget to the top-performing channel
    - take that budget evenly from the other channels
    """
    proposed = current_allocations.copy()
    other_channels = [c for c in CHANNELS if c != top_channel]
    take_from_each = shift_pct / len(other_channels)

    for c in other_channels:
        proposed[c] = proposed[c] - take_from_each

    proposed[top_channel] = proposed[top_channel] + shift_pct
    return proposed


def simulate_day_outcome(day_df: pd.DataFrame, allocations: Dict[str, float]) -> Dict[str, object]:
    """
    Simulates the day's outcome after applying a budget split.

    Simple assumption:
    - if budget increases, impressions/clicks/conversions scale upward
    - if budget decreases, they scale downward

    This keeps the prototype easy to understand.
    """
    total_budget = float(day_df["spend"].sum())
    rows = []

    for _, row in day_df.iterrows():
        channel = row["channel"]
        original_spend = max(float(row["spend"]), 1.0)

        # New budget for this channel based on the chosen allocation.
        new_spend = total_budget * allocations[channel]
        scale = new_spend / original_spend

        # Scale performance using the budget ratio.
        new_impressions = max(1, int(round(float(row["impressions"]) * scale)))
        new_clicks = max(0, int(round(float(row["clicks"]) * scale)))
        new_conversions = max(0, int(round(float(row["conversions"]) * scale)))

        rows.append(
            {
                "date": pd.to_datetime(row["date"]).strftime("%Y-%m-%d"),
                "channel": channel,
                "spend": round(new_spend, 2),
                "impressions": new_impressions,
                "clicks": new_clicks,
                "conversions": new_conversions,
            }
        )

    out_df = pd.DataFrame(rows)
    total_spend = float(out_df["spend"].sum())
    total_impressions = int(out_df["impressions"].sum())
    total_clicks = int(out_df["clicks"].sum())
    total_conversions = int(out_df["conversions"].sum())

    return {
        "rows": rows,
        "totals": {
            "spend": round(total_spend, 2),
            "impressions": total_impressions,
            "clicks": total_clicks,
            "conversions": total_conversions,
            "ctr": total_clicks / total_impressions if total_impressions > 0 else 0.0,
            "cvr": total_conversions / total_clicks if total_clicks > 0 else 0.0,
            "cpa": round(total_spend / total_conversions, 2) if total_conversions > 0 else None,
        },
    }


def summarize_results(results: List[Dict[str, object]]) -> Dict[str, float]:
    """
    Converts the list of daily outcomes into final evaluation metrics.
    """
    df = pd.DataFrame(results)
    total_spend = float(df["spend"].sum())
    total_impressions = int(df["impressions"].sum())
    total_clicks = int(df["clicks"].sum())
    total_conversions = int(df["conversions"].sum())

    return {
        "total_spend": round(total_spend, 2),
        "total_impressions": total_impressions,
        "total_clicks": total_clicks,
        "total_conversions": total_conversions,
        "average_ctr": round(total_clicks / total_impressions, 4) if total_impressions else 0.0,
        "average_cvr": round(total_conversions / total_clicks, 4) if total_clicks else 0.0,
        "average_cpa": round(total_spend / total_conversions, 2) if total_conversions else None,
    }


def format_pct(value: float) -> str:
    """Formats a decimal like 0.0312 as 3.12%."""
    return f"{value * 100:.2f}%"


def format_money(value) -> str:
    """Formats a numeric value as currency, or N/A if missing."""
    if value is None:
        return "N/A"
    return f"${value:,.2f}"

-----

### 6. Define Graph Nodes (Steps)

**Description:**  
Workflow steps for graph runs.

**What each node means:**
- `load_data_node`: reads the CSVs and prepares one dataframe per day
- `metrics_node`: calculates the daily channel metrics
- `reasoning_and_tools_node`: chooses the top channel and prepares actions
- `execution_node`: simulates the day for both the agent and the baseline
- `evaluation_node`: compares final results and writes the explanation



In [ ]:
def load_data_node(state: AgentState) -> AgentState:
    """
    Loads the CSV files and prepares the day-by-day dataset.

    Why this node matters:
    The rest of the graph needs data grouped by date so it can
    simulate one decision cycle per day.
    """
    file_names = ["Google.csv", "Facebook.csv", "Sprouts (Digital Marketing).csv"]

    # Use the read tool to combine all CSVs.
    raw_json = read_performance_data.invoke({"file_names": file_names})

    # StringIO avoids the pandas FutureWarning on literal JSON strings.
    raw_data = pd.read_json(StringIO(raw_json))

    # Create a dictionary like:
    # {
    #   "2025-02-01": dataframe_for_that_day,
    #   "2025-02-02": dataframe_for_that_day,
    #   ...
    # }
    daily_data = {}
    for date_key, group in raw_data.groupby(raw_data["date"].dt.strftime("%Y-%m-%d")):
        daily_data[date_key] = group.reset_index(drop=True)

    dates = sorted(daily_data.keys())

    # Optional debug limiter to keep runs short during testing.
    if MAX_DEBUG_DAYS is not None:
        dates = dates[:MAX_DEBUG_DAYS]
        daily_data = {k: daily_data[k] for k in dates}

    # Start from an equal split across all three channels.
    return {
        "system_prompt": SYSTEM_PROMPT,
        "raw_data": raw_data,
        "daily_data": daily_data,
        "dates": dates,
        "current_day_index": 0,
        "current_allocations": {"Search": 1/3, "Social": 1/3, "Display": 1/3},
        "next_allocations": {"Search": 1/3, "Social": 1/3, "Display": 1/3},
        "decision_history": [],
        "agent_daily_results": [],
        "baseline_daily_results": [],
    }


def metrics_node(state: AgentState) -> AgentState:
    """
    Calculates performance metrics for the current day.

    Why this node matters:
    The agent should not move budget blindly. It first needs to
    measure how each channel performed today.
    """
    date_key = state["dates"][state["current_day_index"]]
    day_df = state["daily_data"][date_key]

    # Use the tool to calculate CTR, CVR, CPA, CPC, and blended_score.
    metrics_json = calculate_channel_metrics.invoke({"day_json": day_df.to_json()})
    metrics = json.loads(metrics_json)

    return {"latest_day_metrics": metrics}


def reasoning_and_tools_node(state: AgentState) -> AgentState:
    """
    Decides where tomorrow's extra budget should go.

    Why this node matters:
    This is the "brain" of the workflow.
    It either:
    - uses a local heuristic in debug mode, or
    - uses the LLM for structured reasoning

    After choosing a top channel, it applies guardrails and
    creates simulated tool actions.
    """
    date_key = state["dates"][state["current_day_index"]]
    metrics = state["latest_day_metrics"]
    current_allocations = state["current_allocations"]

    if DEBUG_MODE:
        # Debug mode:
        # Choose the top channel using the blended score directly.
        ranked = sorted(metrics.items(), key=lambda x: x[1]["blended_score"], reverse=True)
        top_channel = ranked[0][0]

        decision_dict = {
            "top_channel": top_channel,
            "metric_used": "blended_score",
            "reasoning": (
                f"{top_channel} was chosen because it had the strongest blended score "
                f"based on CTR, CVR, and CPA for {date_key}."
            ),
            "suggested_actions": ["adjust_bid"],
        }
    else:
        # Live LLM mode:
        # Ask the model to reason over the current allocations and metrics.
        prompt = f"""
Today is {date_key}.

Current budget allocations:
{json.dumps(current_allocations, indent=2)}

Current channel metrics:
{json.dumps(metrics, indent=2)}

Please choose the best channel for tomorrow's extra budget.
Use simple marketing reasoning.
Prefer conversions, CVR, CTR, and CPA together.
Also suggest any useful actions from:
- adjust_bid
- pause_ad_group
- request_new_creatives

Return a small structured decision.
""".strip()

        structured_llm = llm.with_structured_output(BudgetDecision)
        decision = structured_llm.invoke(
            [
                SystemMessage(content=state["system_prompt"]),
                HumanMessage(content=prompt),
            ]
        )
        decision_dict = decision.model_dump()

    # First apply the explore/exploit move:
    # shift more budget toward the top channel.
    proposed = heuristic_budget_shift(
        current_allocations=current_allocations,
        top_channel=decision_dict["top_channel"],
        shift_pct=DAILY_SHIFT,
    )

    # Then apply guardrails so the move is safe and stable.
    next_allocations = apply_guardrails(
        old_allocations=current_allocations,
        proposed_allocations=proposed,
    )

    # Simulated action log.
    tool_actions = []

    # Main action: increase the bid for the chosen channel.
    tool_actions.append(
        json.loads(
            adjust_bid.invoke(
                {
                    "channel": decision_dict["top_channel"],
                    "direction": "up",
                    "percent": 10.0,
                    "reason": decision_dict["reasoning"],
                }
            )
        )
    )

    # Find the weakest channel to decide whether extra actions make sense.
    weak_channels = sorted(CHANNELS, key=lambda c: metrics[c]["blended_score"])
    weakest = weak_channels[0]
    weakest_ctr = metrics[weakest]["ctr"]
    weakest_cvr = metrics[weakest]["cvr"]

    # If CTR is weak, ask for new creatives.
    if weakest != decision_dict["top_channel"] and weakest_ctr < 0.02:
        tool_actions.append(
            json.loads(
                request_new_creatives.invoke(
                    {
                        "channel": weakest,
                        "reason": f"{weakest} has a weak CTR and needs fresh creative testing.",
                    }
                )
            )
        )

    # If conversion quality is weak, simulate pausing a bad ad group.
    if weakest != decision_dict["top_channel"] and weakest_cvr < 0.04:
        tool_actions.append(
            json.loads(
                pause_ad_group.invoke(
                    {
                        "channel": weakest,
                        "reason": f"{weakest} has weak conversion efficiency and one weak ad group should be reviewed.",
                    }
                )
            )
        )

    return {
        "latest_llm_decision": decision_dict,
        "latest_tool_actions": tool_actions,
        "next_allocations": next_allocations,
    }


def execution_node(state: AgentState) -> AgentState:
    """
    Simulates the day under two strategies:
    1. the agent's chosen budget allocation
    2. the equal-split baseline

    Why this node matters:
    The assignment asks for evaluation. This node collects
    the day-by-day outcomes needed for that comparison.
    """
    date_key = state["dates"][state["current_day_index"]]
    day_df = state["daily_data"][date_key]

    agent_outcome = simulate_day_outcome(day_df, state["current_allocations"])
    baseline_outcome = simulate_day_outcome(
        day_df,
        {"Search": 1/3, "Social": 1/3, "Display": 1/3}
    )

    # Keep a detailed log so each decision is auditable.
    history_row = {
        "date": date_key,
        "allocations_used_today": {k: round(v, 4) for k, v in state["current_allocations"].items()},
        "next_allocations": {k: round(v, 4) for k, v in state["next_allocations"].items()},
        "llm_decision": state["latest_llm_decision"],
        "tool_actions": state["latest_tool_actions"],
        "metrics": state["latest_day_metrics"],
        "agent_totals": agent_outcome["totals"],
        "baseline_totals": baseline_outcome["totals"],
    }

    return {
        "agent_daily_results": state["agent_daily_results"] + [agent_outcome["totals"] | {"date": date_key}],
        "baseline_daily_results": state["baseline_daily_results"] + [baseline_outcome["totals"] | {"date": date_key}],
        "decision_history": state["decision_history"] + [history_row],
        "current_allocations": state["next_allocations"],
        "current_day_index": state["current_day_index"] + 1,
    }


def evaluation_node(state: AgentState) -> AgentState:
    """
    Creates the final comparison between the agent and the baseline.

    Why this node matters:
    The prototype should not only make decisions. It should also
    explain whether those decisions improved results.
    """
    agent_summary = summarize_results(state["agent_daily_results"])
    baseline_summary = summarize_results(state["baseline_daily_results"])

    explanation_parts = []

    explanation_parts.append(
        "Heuristic and explore/exploit: the agent starts with an equal split across Search, Social, and Display. "
        "Each day it calculates CTR, CVR, and CPA for each channel, then shifts 10% more budget toward the strongest channel. "
        "At the same time, it keeps budget on the other channels so they can continue gathering data and remain part of the learning loop."
    )

    explanation_parts.append(
        "Guardrails: the workflow caps per-day budget changes at plus or minus 20%, keeps a 20% minimum floor for every channel, "
        "and never fully shuts a channel off. These rules reduce overreaction to one strong or weak day and make the daily allocation changes easier to explain and audit."
    )

    if agent_summary["total_conversions"] > baseline_summary["total_conversions"]:
        explanation_parts.append(
            f"In this run, the agent beat the equal-split baseline on total conversions "
            f"({agent_summary['total_conversions']} vs {baseline_summary['total_conversions']})."
        )
    else:
        explanation_parts.append(
            f"In this run, the equal-split baseline matched or beat the agent on total conversions "
            f"({baseline_summary['total_conversions']} vs {agent_summary['total_conversions']})."
        )

    if agent_summary["average_cpa"] is not None and baseline_summary["average_cpa"] is not None:
        if agent_summary["average_cpa"] < baseline_summary["average_cpa"]:
            explanation_parts.append(
                f"The agent was also more cost-efficient because its average CPA was lower "
                f"({agent_summary['average_cpa']} vs {baseline_summary['average_cpa']})."
            )
        elif agent_summary["average_cpa"] > baseline_summary["average_cpa"]:
            explanation_parts.append(
                f"The baseline was more cost-efficient because its average CPA was lower "
                f"({baseline_summary['average_cpa']} vs {agent_summary['average_cpa']})."
            )
        else:
            explanation_parts.append("Both strategies had the same average CPA.")

    if agent_summary["average_ctr"] > baseline_summary["average_ctr"]:
        explanation_parts.append(
            f"The agent also improved average CTR ({agent_summary['average_ctr']} vs {baseline_summary['average_ctr']}), "
            "which suggests budget was pulled toward channels with stronger engagement."
        )
    elif agent_summary["average_ctr"] < baseline_summary["average_ctr"]:
        explanation_parts.append(
            f"The baseline had the better average CTR ({baseline_summary['average_ctr']} vs {agent_summary['average_ctr']})."
        )
    else:
        explanation_parts.append("Both strategies had the same average CTR.")

    explanation_parts.append(
        "The main evaluation metrics are total conversions, total clicks, average CTR, average CVR, "
        "and average CPA. Together these show not only whether the agent generated more results, "
        "but also whether it did so efficiently."
    )

    if DEBUG_MODE:
        explanation_parts.append(
            "This run used debug mode for reasoning, which means the graph and tools still ran, "
            "but the daily decision step used a local heuristic instead of repeated live API calls."
        )

    return {
        "evaluation_summary": {
            "agent": agent_summary,
            "baseline_equal_split": baseline_summary,
            "explanation": {"text": " ".join(explanation_parts)},
        }
    }


def route_after_load(state: AgentState) -> str:
    """
    Sends the graph from the load step into the metrics step.
    """
    return "metrics"


def route_after_execute(state: AgentState) -> str:
    """
    Decides whether the graph should:
    - continue to the next day, or
    - stop daily looping and run the final evaluation
    """
    if state["current_day_index"] >= len(state["dates"]):
        return "evaluate"
    return "metrics"

-----

### 7. Build and Run the Graph

**Description:**  
This is where the workflow is assembled using `StateGraph`, just like in the template.

**How the flow works (in order):**
1. load the data
2. calculate metrics
3. reason and choose actions
4. execute the decision
5. loop until all days are processed
6. evaluate final results


In [ ]:
# Create the workflow object using our shared state type.
graph = StateGraph(AgentState)

# Add each node (step) into the graph.
graph.add_node("load_data", load_data_node)
graph.add_node("metrics", metrics_node)
graph.add_node("reasoning_and_tools", reasoning_and_tools_node)
graph.add_node("execute", execution_node)
graph.add_node("evaluate", evaluation_node)

# Define the starting point.
graph.set_entry_point("load_data")

# After data is loaded, always go to metrics.
graph.add_conditional_edges("load_data", route_after_load, {"metrics": "metrics"})

# Main daily loop:
# metrics -> reasoning -> execution
graph.add_edge("metrics", "reasoning_and_tools")
graph.add_edge("reasoning_and_tools", "execute")

# After execution, either:
# - go back to metrics for the next day, or
# - go to evaluation if all dates are done
graph.add_conditional_edges(
    "execute",
    route_after_execute,
    {"metrics": "metrics", "evaluate": "evaluate"},
)

# End the workflow after evaluation.
graph.add_edge("evaluate", END)

# Compile the graph so it can be invoked.
app = graph.compile()

print("Graph compiled successfully.")

Graph compiled successfully.


-----

### 8. Run the Prototype Agent and Print Results

**Description:**  
This runs the full workflow and prints a clean summary.

**What it shows for output:**
- final results for the agent
- final results for the equal-split baseline
- a detailed explanation of why the results happened
- the first 5 daily decisions with reason strings and tool actions


In [ ]:
# Run the full LangGraph workflow.
final_state = app.invoke({})

# Pull out the two main summaries for easier printing.
agent = final_state["evaluation_summary"]["agent"]
baseline = final_state["evaluation_summary"]["baseline_equal_split"]

print("=" * 72)
print("DECISION PREVIEW WITH DAILY BUDGETS (FIRST 5 DAYS)")
print("=" * 72)

for row in final_state["decision_history"][:5]:
    date = row["date"]
    agent_total_spend = row["agent_totals"]["spend"]
    used_alloc = row["allocations_used_today"]
    next_alloc = row["next_allocations"]
    llm_decision = row["llm_decision"]
    metrics = row["metrics"]
    tool_actions = row["tool_actions"]

    print(f"Date: {date}")
    print("  Today's allocated budget by channel:")

    for ch in ["Search", "Social", "Display"]:
        budget_amount = agent_total_spend * used_alloc[ch]
        print(
            f"    - {ch} ({CHANNEL_LABELS.get(ch,'')}): {format_money(budget_amount)} "
            f"({used_alloc[ch] * 100:.1f}% of total budget)"
        )

    print("  Tomorrow's planned allocation by channel:")
    for ch in ["Search", "Social", "Display"]:
        next_budget_amount = agent_total_spend * next_alloc[ch]
        print(
            f"    - {ch} ({CHANNEL_LABELS.get(ch,'')}): {format_money(next_budget_amount)} "
            f"({next_alloc[ch] * 100:.1f}% planned share)"
        )

    print(f"  Top channel chosen: {llm_decision['top_channel']}")
    print(f"  Main metric used:   {llm_decision['metric_used']}")
    print(f"  Reason:             {llm_decision['reasoning']}")

    print("  Channel status report:")
    for ch in ["Search", "Social", "Display"]:
        ch_metrics = metrics[ch]
        status_bits = [
            f"CTR {format_pct(ch_metrics['ctr'])}",
            f"CVR {format_pct(ch_metrics['cvr'])}",
            f"CPA {format_money(ch_metrics['cpa'])}",
        ]

        action_notes = []
        for action in tool_actions:
            if action["channel"] == ch:
                if action["tool"] == "adjust_bid":
                    action_notes.append(
                        f"bid adjusted {action['direction']} by {action['percent']:.0f}% because {action['reason']}"
                    )
                elif action["tool"] == "request_new_creatives":
                    action_notes.append(
                        "new creatives requested to improve weak CTR and test stronger ad copy or visuals"
                    )
                elif action["tool"] == "pause_ad_group":
                    action_notes.append(
                        "one weak ad group flagged to pause because conversion efficiency was poor"
                    )

        if not action_notes:
            if ch_metrics["ctr"] < 0.02:
                action_notes.append("budget kept active but monitored closely because CTR is weak")
            else:
                action_notes.append("channel remains active with no extra intervention today")

        print(f"    - {ch} ({CHANNEL_LABELS.get(ch,'')}): " + "; ".join(status_bits) + ". Action: " + " | ".join(action_notes))

    print(f"  Agent conversions today:    {row['agent_totals']['conversions']}")
    print(f"  Baseline conversions today: {row['baseline_totals']['conversions']}")
    print("-" * 72)

print("\n" + "=" * 72)
print("WHY THE RESULTS LOOK THIS WAY")
print("=" * 72)

explanation_text = final_state["evaluation_summary"]["explanation"]["text"]
sentences = [s.strip() for s in explanation_text.split(". ") if s.strip()]
paragraph_lines = []
current = ""

for s in sentences:
    sentence = s if s.endswith(".") else s + "."
    if not current:
        current = sentence
    elif len(current) + len(sentence) + 1 <= 88:
        current = current + " " + sentence
    else:
        paragraph_lines.append(current)
        current = sentence

if current:
    paragraph_lines.append(current)

for line in paragraph_lines:
    print(line)
print()

print("=" * 72)
print("GUARDRAILS, HEURISTICS, AND FORWARD EVALUATION")
print("=" * 72)
print("Heuristic / explore-exploit:")
print("  - Start with an equal split across Search, Social, and Display.")
print("  - Each day, compute CTR, CVR, and CPA for each channel.")
print("  - Shift 10% more budget toward the top performer while keeping budget on the others.")
print()
print("Guardrails:")
print("  - Per-day budget change is capped at +/-20%.")
print("  - Every channel keeps at least a 20% budget floor.")
print("  - No channel is fully shut off, so learning continues across all channels.")
print("  - Each day's decision is logged with a plain-English reason string.")
print()
print("How evaluation is judged moving forward:")
print("  - Compare total conversions against the equal-split baseline.")
print("  - Compare average CPA to see whether conversions are efficient.")
print("  - Compare average CTR and average CVR to see whether engagement and quality improve.")
print("  - Review the daily logs to see whether the budget moves stay stable and explainable.")

print("\n" + "=" * 72)
print("FINAL RESULTS")
print("=" * 72)

print("AGENT STRATEGY")
print(f"  Total spend:        {format_money(agent['total_spend'])}")
print(f"  Total impressions:  {agent['total_impressions']}")
print(f"  Total clicks:       {agent['total_clicks']}")
print(f"  Total conversions:  {agent['total_conversions']}")
print(f"  Average CTR:        {format_pct(agent['average_ctr'])}")
print(f"  Average CVR:        {format_pct(agent['average_cvr'])}")
print(f"  Average CPA:        {format_money(agent['average_cpa'])}")

print("\nEQUAL-SPLIT BASELINE")
print(f"  Total spend:        {format_money(baseline['total_spend'])}")
print(f"  Total impressions:  {baseline['total_impressions']}")
print(f"  Total clicks:       {baseline['total_clicks']}")
print(f"  Total conversions:  {baseline['total_conversions']}")
print(f"  Average CTR:        {format_pct(baseline['average_ctr'])}")
print(f"  Average CVR:        {format_pct(baseline['average_cvr'])}")
print(f"  Average CPA:        {format_money(baseline['average_cpa'])}")

DECISION PREVIEW WITH DAILY BUDGETS (FIRST 5 DAYS)
Date: 2025-02-01
  Today's allocated budget by channel:
    - Search (Google): $183.98 (33.3% of total budget)
    - Social (Facebook): $183.98 (33.3% of total budget)
    - Display (Sprouts): $183.98 (33.3% of total budget)
  Tomorrow's planned allocation by channel:
    - Search (Google): $239.18 (43.3% planned share)
    - Social (Facebook): $156.38 (28.3% planned share)
    - Display (Sprouts): $156.38 (28.3% planned share)
  Top channel chosen: Search
  Main metric used:   blended_score
  Reason:             Search was chosen because it had the strongest blended score based on CTR, CVR, and CPA for 2025-02-01.
  Channel status report:
    - Search (Google): CTR 4.63%; CVR 9.64%; CPA $4.51. Action: bid adjusted up by 10% because Search was chosen because it had the strongest blended score based on CTR, CVR, and CPA for 2025-02-01.
    - Social (Facebook): CTR 2.82%; CVR 6.11%; CPA $12.21. Action: channel remains active with no extr

-----

### 9. Save Evaluation Log (OPTIONAL)

**Description:**  
Decision logs & summary are exported as separate files for future reference (if needed).

**What gets saved - added to the files left-side taskbar:**
- a detailed daily decision log
- a final evaluation summary


In [ ]:
decision_rows = []

# Flatten the decision history into a CSV-friendly structure.
for row in final_state["decision_history"]:
    decision_rows.append(
        {
            "date": row["date"],
            "search_alloc_today": row["allocations_used_today"]["Search"],
            "social_alloc_today": row["allocations_used_today"]["Social"],
            "display_alloc_today": row["allocations_used_today"]["Display"],
            "search_alloc_next": row["next_allocations"]["Search"],
            "social_alloc_next": row["next_allocations"]["Social"],
            "display_alloc_next": row["next_allocations"]["Display"],
            "top_channel": row["llm_decision"]["top_channel"],
            "metric_used": row["llm_decision"]["metric_used"],
            "reason": row["llm_decision"]["reasoning"],
            "tool_actions": json.dumps(row["tool_actions"]),
            "agent_conversions": row["agent_totals"]["conversions"],
            "baseline_conversions": row["baseline_totals"]["conversions"],
            "agent_ctr": row["agent_totals"]["ctr"],
            "baseline_ctr": row["baseline_totals"]["ctr"],
        }
    )

decision_log_df = pd.DataFrame(decision_rows)

evaluation_df = pd.DataFrame(
    [
        {"strategy": "agent", **final_state["evaluation_summary"]["agent"]},
        {"strategy": "baseline_equal_split", **final_state["evaluation_summary"]["baseline_equal_split"]},
    ]
)

decision_log_df.to_csv("ad_agent_decision_log_annotated.csv", index=False)
evaluation_df.to_csv("ad_agent_evaluation_summary_annotated.csv", index=False)

print("Saved files:")
print("- ad_agent_decision_log_annotated.csv")
print("- ad_agent_evaluation_summary_annotated.csv")

Saved files:
- ad_agent_decision_log_annotated.csv
- ad_agent_evaluation_summary_annotated.csv
